# 01 — Exploration des données collectées

Vérifie que le pipeline Mastodon → Kafka → Spark → PostgreSQL alimente bien la base.

In [4]:
import sys, json
sys.path.insert(0, "/home/jovyan/work")
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from common import config

engine = create_engine(config.SQLALCHEMY_URL)
q = lambda sql: pd.read_sql(sql, engine)

# Palette : positif = bleu, négatif = orange (paire distinguable par les daltoniens)
COLORS = {"positive": "#2a78d6", "negative": "#eb6834"}
sns.set_theme(style="whitegrid", rc={"axes.spines.top": False, "axes.spines.right": False,
                                     "grid.color": "#e6e6e6", "axes.titleweight": "bold"})
FIG_DIR = config.OUTPUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()


## Volume et période couverte

In [5]:
q("""SELECT COUNT(*) AS nb_toots, MIN(created_at) AS debut, MAX(created_at) AS fin,
          COUNT(DISTINCT account) AS nb_comptes FROM toots""")

,nb_toots,debut,fin,nb_comptes
0,935,2026-10-09 10:45:00,2026-10-11 21:55:33,411


## Échantillon de toots enrichis

In [6]:
q("SELECT created_at, account, clean_text, hashtags, sentiment, ROUND(score_pos::numeric, 3) AS score_pos FROM toots ORDER BY created_at DESC LIMIT 10")

,created_at,account,clean_text,hashtags,sentiment,score_pos
0,2026-10-11 21:55:33,earthalerts,km w of gorontalo indonesia sun oct utc m eart...,"earthquake,seismic,earthalerts,disasteralert,b...",positive,0.525
1,2026-10-11 21:55:13,hans@mastodon.crazynewworld.net,apple tv the new apple tv could have a built i...,"apple,llm,news,bot",positive,0.650
2,2026-10-11 21:55:09,hans@mastodon.crazynewworld.net,homepod mini new apple tv k homeview and homep...,"apple,llm,news,bot",positive,0.706
3,2026-10-11 21:55:03,variety_feed,john stamos addresses mike love s health as be...,"variety,news,aljardine,johnstamos,mikelove,the...",positive,1.000
4,2026-10-11 21:54:08,PeterKReilly@mstdn.party,sorry a very late caturday submission cats cat...,"caturday,cats,catsofmastodon,photography,animals",negative,0.427
5,2026-10-11 21:54:04,AITrendsNews,top ai github repos blowing up right now oct s...,"shorts,ai,github,opensource,machinelearning,to...",positive,0.969
6,2026-10-11 21:54:02,hermes_vector,russia strikes kharkiv power plant after trump...,"world,news",positive,1.000
7,2026-10-11 21:54:01,hermes_vector,trump and rubio consider firing guilfoyle over...,"politics,news",negative,0.096
8,2026-10-11 21:53:52,AITrendsNews,top ai github repos blowing up right now oct h...,"ai,github,opensource,machinelearning,toprepos",positive,0.959
9,2026-10-11 21:53:20,techwire@gamefan.net,rd yandex cloud data center was hit article ur...,"tech,technology,technews,ai,gadgets,software,c...",positive,0.984


## Toots les plus positifs / négatifs (selon le modèle)

In [4]:
display(q("SELECT clean_text, score_pos FROM toots ORDER BY score_pos DESC LIMIT 5"))
display(q("SELECT clean_text, score_pos FROM toots ORDER BY score_pos ASC LIMIT 5"))

,clean_text,score_pos


,clean_text,score_pos


## Agrégats streaming (fenêtres de 5 min)

In [7]:
q("SELECT * FROM sentiment_window ORDER BY window_start DESC LIMIT 10")

,window_start,window_end,sentiment,nb_toots,avg_score
0,2026-10-11 21:40:00,2026-10-11 21:45:00,negative,7,0.110090
1,2026-10-11 21:40:00,2026-10-11 21:45:00,positive,19,0.800652
2,2026-10-11 21:35:00,2026-10-11 21:40:00,positive,14,0.876880
3,2026-10-11 21:35:00,2026-10-11 21:40:00,negative,7,0.054627
4,2026-10-11 21:30:00,2026-10-11 21:35:00,positive,11,0.852013
5,2026-10-11 21:30:00,2026-10-11 21:35:00,negative,7,0.288527
6,2026-10-11 21:25:00,2026-10-11 21:30:00,positive,12,0.765288
7,2026-10-11 21:25:00,2026-10-11 21:30:00,negative,1,0.273875
8,2026-10-11 21:20:00,2026-10-11 21:25:00,positive,9,0.859978
9,2026-10-11 21:20:00,2026-10-11 21:25:00,negative,1,0.040823


In [8]:
q("SELECT hashtag, SUM(nb_occurrences) AS total FROM hashtag_window GROUP BY 1 ORDER BY 2 DESC LIMIT 10")

,hashtag,total
0,ai,208.0
1,news,207.0
2,tech,128.0
3,software,111.0
4,gaming,91.0
5,science,88.0
6,music,87.0
7,technology,83.0
8,programming,73.0
9,books,69.0


## Même lecture avec Spark (JDBC + Spark SQL)

In [9]:
spark = config.get_spark("notebook-exploration", postgres=True, ui_port=4042)
toots = spark.read.jdbc(config.JDBC_URL, "toots", properties=config.JDBC_PROPS)
toots.createOrReplaceTempView("toots")
spark.sql("""
    SELECT hour(created_at) AS heure, sentiment, COUNT(*) AS nb
    FROM toots GROUP BY 1, 2 ORDER BY 1, 2
""").show(48)

+-----+---------+---+
|heure|sentiment| nb|
+-----+---------+---+
|    1| negative|  1|
|    5| positive|  1|
|    6| positive|  1|
|    7| negative|  2|
|    8| positive|  1|
|    9| positive|  1|
|   10| negative|  9|
|   11| negative|  5|
|   11| positive|  3|
|   12| negative|  2|
|   12| positive|  7|
|   13| negative| 11|
|   13| positive| 10|
|   14| negative|  7|
|   14| positive| 10|
|   15| negative|  5|
|   15| positive| 22|
|   16| negative| 13|
|   16| positive| 23|
|   17| negative| 26|
|   17| positive| 54|
|   18| negative| 90|
|   18| positive|167|
|   19| negative| 72|
|   19| positive|147|
|   20| negative| 26|
|   20| positive| 41|
|   21| negative| 49|
|   21| positive|129|
+-----+---------+---+



In [8]:
spark.stop()